# metrics

> Pixel-level overlap metrics (per-class IoU/Dice/precision/recall) and object-level detection metrics (connected-component hit/miss/false-alarm).

Pixel metrics tell you how well the shapes overlap; object metrics tell you how many distinct targets you actually found. In inspection a missed object is usually far costlier than a few wrong pixels, so we track both.

In [ ]:
#| default_exp metrics

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from statistics import mean
from typing import Dict, List

import cv2
import numpy as np
import torch

## Pixel metrics

We accumulate a confusion tally (TP/FP/FN) per class across an epoch, then compute overlap metrics once. Binary (`num_classes == 1`) thresholds the sigmoid for the foreground class; multi-class takes the argmax and averages metrics over foreground classes (background excluded from the mean).

In [ ]:
# experiment: IoU of a perfect vs an off-by-one prediction
gt = torch.zeros(1, 1, 8, 8); gt[..., 2:6, 2:6] = 1
perfect = (gt > 0.5)
inter = (perfect & (gt > 0.5)).sum().item(); union = (perfect | (gt > 0.5)).sum().item()
print('perfect IoU:', inter / union)

In [ ]:
#| export
class SegMetrics:
    """Accumulate per-class pixel metrics over batches.

    Example:
        >>> m = SegMetrics(num_classes=1)
        >>> gt = torch.zeros(1, 8, 8); gt[2:6, 2:6] = 1
        >>> logits = torch.where(gt.bool(), 10.0, -10.0)[None]  # (1,1,8,8)
        >>> m.update(logits, gt[None]); round(m.compute()['iou'], 3)
        1.0
    """

    def __init__(self, num_classes: int = 1, threshold: float = 0.5):
        self.num_classes = num_classes
        self.threshold = threshold
        self.labels = [1] if num_classes == 1 else list(range(num_classes))
        self.reset()

    def reset(self) -> None:
        self.tp = {l: 0 for l in self.labels}
        self.fp = {l: 0 for l in self.labels}
        self.fn = {l: 0 for l in self.labels}

    @torch.no_grad()
    def update(self, logits: torch.Tensor, targets: torch.Tensor) -> None:
        "logits: (B, C, H, W); targets: (B, H, W) labels (or (B,1,H,W) for binary)."
        if self.num_classes == 1:
            preds = (torch.sigmoid(logits)[:, 0] >= self.threshold).long()
            gts = targets.long().squeeze(1) if targets.dim() == 4 else targets.long()
        else:
            preds = logits.argmax(1)
            gts = targets.long()
        for l in self.labels:
            p, t = (preds == l), (gts == l)
            self.tp[l] += int((p & t).sum())
            self.fp[l] += int((p & ~t).sum())
            self.fn[l] += int((~p & t).sum())

    def compute(self) -> Dict:
        "Per-class metrics plus their mean (multi-class excludes background 0)."
        eps = 1e-6
        mean_labels = [l for l in self.labels if not (self.num_classes > 1 and l == 0)]
        per, agg = {}, {'iou': [], 'dice': [], 'precision': [], 'recall': []}
        for l in self.labels:
            tp, fp, fn = self.tp[l], self.fp[l], self.fn[l]
            scores = {
                'iou': tp / (tp + fp + fn + eps),
                'dice': 2 * tp / (2 * tp + fp + fn + eps),
                'precision': tp / (tp + fp + eps),
                'recall': tp / (tp + fn + eps),
            }
            per[l] = scores
            if l in mean_labels:
                for k in agg:
                    agg[k].append(scores[k])
        out = {k: mean(v) for k, v in agg.items()}
        out['per_class'] = per
        return out

In [ ]:
# test: perfect binary prediction -> IoU 1; multi-class shape is right
gt = torch.zeros(1, 8, 8); gt[0, 2:6, 2:6] = 1
logits = torch.where(gt.bool(), 10.0, -10.0)[None]
m = SegMetrics(num_classes=1); m.update(logits, gt[None])
assert round(m.compute()['iou'], 3) == 1.0
mm = SegMetrics(num_classes=3); mm.update(torch.randn(2, 3, 8, 8), torch.randint(0, 3, (2, 8, 8)))
assert set(mm.compute()['per_class'].keys()) == {0, 1, 2}

## Object-level detection

We label connected components in the predicted and ground-truth foreground, then count a ground-truth object as *detected* if enough of it is covered, and a predicted object as a *false alarm* if it barely overlaps any target.

In [ ]:
#| export
def object_metrics(
    pred_mask: np.ndarray,        # (H, W) binary prediction
    gt_mask: np.ndarray,          # (H, W) binary ground truth
    overlap_thresh: float = 0.5,  # fraction of an object that must overlap to count
) -> Dict:
    """Connected-component detection stats for one image.

    Returns counts of total/detected/missed ground-truth objects and false alarms.
    """
    pred_bin = (pred_mask > 0).astype(np.uint8)
    gt_bin = (gt_mask > 0).astype(np.uint8)
    n_gt, gt_labels = cv2.connectedComponents(gt_bin)
    n_pred, pred_labels = cv2.connectedComponents(pred_bin)
    n_gt, n_pred = n_gt - 1, n_pred - 1
    detected = sum(
        ((gt_labels == i) & (pred_bin > 0)).sum() / ((gt_labels == i).sum() + 1e-6) >= overlap_thresh
        for i in range(1, n_gt + 1)
    )
    false_alarms = sum(
        ((pred_labels == j) & (gt_bin > 0)).sum() / ((pred_labels == j).sum() + 1e-6) < overlap_thresh
        for j in range(1, n_pred + 1)
    )
    return {'total_gt': n_gt, 'detected': int(detected), 'missed': n_gt - int(detected),
            'false_alarms': int(false_alarms),
            'detection_rate': int(detected) / (n_gt + 1e-6)}


class ObjectMetrics:
    "Accumulate `object_metrics` across many images."

    def __init__(self, overlap_thresh: float = 0.5):
        self.overlap_thresh = overlap_thresh
        self.reset()

    def reset(self) -> None:
        self.total_gt = self.detected = self.false_alarms = 0

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray) -> None:
        r = object_metrics(pred_mask, gt_mask, self.overlap_thresh)
        self.total_gt += r['total_gt']; self.detected += r['detected']; self.false_alarms += r['false_alarms']

    def compute(self) -> Dict:
        missed = self.total_gt - self.detected
        return {'total_objects': self.total_gt, 'detected': self.detected, 'missed': missed,
                'detection_rate': self.detected / (self.total_gt + 1e-6),
                'false_alarms': self.false_alarms}

In [ ]:
# test: two separate gt blobs, both predicted -> detection_rate 1.0, no false alarms
gt = np.zeros((40, 40), np.uint8); gt[5:15, 5:15] = 1; gt[25:35, 25:35] = 1
r = object_metrics(gt, gt)
assert r['total_gt'] == 2 and r['detected'] == 2 and r['false_alarms'] == 0
acc = ObjectMetrics(); acc.update(gt, gt)
assert acc.compute()['detection_rate'] == r['detection_rate']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()